# 07 — Student nhỏ: học nhãn thật và học thêm Teacher

Phụ trách: Phúc · Tuần 7. Đọc từng cell từ trên xuống.
Đây là notebook giải thích thí nghiệm development, không phải final Test hoặc Android release.
Teacher v2 giữ nguyên. Chạy `python scripts/train_student.py` từ thư mục repo để tạo targets/artifacts trước.
Notebook này không xuất đè artifact; phần fit bên dưới chỉ tái hiện hai mô hình nhỏ để hiểu flow.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from scipy.sparse import vstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))
from train_student import load_data
from phishguard_vn.student import metrics, choose_threshold, load_student

config = json.loads((ROOT / "configs/week7_student.json").read_text(encoding="utf-8"))
train, val, excluded = load_data(ROOT)
pd.DataFrame({"split": ["Train", "Validation"],
              "rows": [len(train), len(val)],
              "phishing": [train.target.sum(), val.target.sum()]})

,split,rows,phishing
0,Train,13078,2074
1,Validation,1585,24


## 1. Teacher targets là gì?

Teacher đã dự đoán trên các mẫu Train. Ta chỉ lấy file theo ID, không fit Teacher ở notebook.
`target` là nhãn thật; `teacher_score` là score chưa calibrated.
Bản KD học q = (1-alpha)*nhãn thật + alpha*score Teacher. Teacher đã thấy Train nên đây là
targets in-sample, không phải OOF. Validation không tham gia fitting.

In [2]:
targets = pd.read_csv(ROOT / "artifacts/student_week7/teacher_targets.csv.gz")
targets = train[["id"]].merge(targets, on="id", validate="one_to_one")
assert targets.id.tolist() == train.id.tolist()
assert len(targets) == len(train)
targets.head()  # Chỉ có ID/nhãn/score, không chứa URL thật

,id,target,teacher_score,teacher_label,kd_target
0,0004e6cc-1580-503e-8ac9-b8039a0ebee9,0,0.015480,0,0.007740
1,00091f3e-1d47-52cb-b645-2a4c91788ba0,0,0.004145,0,0.002072
2,000c7930-c2e6-52c8-aee2-1cd639aadfc4,0,0.001806,0,0.000903
3,000f5203-2e84-50ed-b16b-738c5d96c199,0,0.000345,0,0.000172
4,001277ac-004d-5d93-9e5c-b718397e5ddd,0,0.024184,0,0.012092


## 2. Từ URL thành số bằng TF-IDF ký tự

`train.text` là URL đã canonicalize. Tách các đoạn 3–5 ký tự, chọn tối đa 8.192 cột.
Vocabulary và IDF chỉ học từ Train; Validation chỉ transform.
Hai cách học dùng chung ma trận để so sánh công bằng.

In [3]:
vectorizer = TfidfVectorizer(
    analyzer="char", ngram_range=tuple(config["ngram_range"]),
    min_df=config["min_df"], max_features=config["max_features"],
    lowercase=False, sublinear_tf=True, norm="l2",
)
X_train = vectorizer.fit_transform(train.text)
X_val = vectorizer.transform(val.text)
print("Train:", X_train.shape, "Validation:", X_val.shape)

Train: (13078, 8192) Validation: (1585, 8192)


## 3. Gọi mô hình và train trực tiếp

Với mỗi mẫu: bản nhãn 1 có trọng số q, bản nhãn 0 có trọng số 1-q.
Nếu q là 0/1 thì tương đương học nhãn thật. Nếu q nằm giữa 0 và 1 thì mô hình học mục tiêu mềm.
Tổng trọng số mỗi mẫu luôn bằng 1. Hai bản cùng C, solver, seed và vocabulary.

In [4]:
results = []
for name in ("hard_only", "kd_mixed"):
    q = train.target.to_numpy() if name == "hard_only" else targets.kd_target.to_numpy()
    model = LogisticRegression(
        C=config["C"], solver="liblinear",
        max_iter=config["max_iter"], random_state=config["seed"],
    )
    model.fit(
        vstack([X_train, X_train]),
        np.r_[np.ones(len(q)), np.zeros(len(q))],
        sample_weight=np.r_[q, 1-q],
    )
    score = model.predict_proba(X_val)[:, 1]
    threshold = choose_threshold(val.target, score, config["fpr_budget"])
    results.append({"candidate": name, **metrics(val.target, score, threshold)})
pd.DataFrame(results)

,candidate,ap,precision,recall,f1,fpr,tp,fp,tn,fn,threshold
0,hard_only,0.292578,0.363636,0.333333,0.347826,0.008969,8,14,1547,16,0.548606
1,kd_mixed,0.304568,0.380952,0.333333,0.355556,0.008328,8,13,1548,16,0.564856


## 4. Đọc kết quả đúng phạm vi

Validation được dùng chọn model/ngưỡng, nên bảng trên không phải final performance độc lập.
Chỉ có 24 phishing trong Validation. So thêm với Teacher đã phát hành; agreement cao có thể
đến từ đa số benign. Không chạy Official Test hoặc external ở notebook này.

In [5]:
teacher = json.loads((ROOT / "reports/tables/student_week7/teacher_validation.json").read_text(encoding="utf-8"))
pd.DataFrame([teacher, *results])

,candidate,ap,precision,recall,f1,fpr,tp,fp,tn,fn,threshold
0,released_teacher_v2,0.554910,0.517241,0.625000,0.566038,0.008969,15,14,1547,9,0.694574
1,hard_only,0.292578,0.363636,0.333333,0.347826,0.008969,8,14,1547,16,0.548606
2,kd_mixed,0.304568,0.380952,0.333333,0.355556,0.008328,8,13,1548,16,0.564856


## 5. Gọi Student đã xuất, không cần Teacher

JSON lưu vocabulary, IDF, trọng số, intercept và threshold.
Đây là Python reference cho Hy port Dart; chưa có Android parity hoặc số đo thiết bị.
Teacher v2 vẫn là model mặc định của ứng dụng.

In [6]:
student = load_student(ROOT)
student.predict_one("https://EXAMPLE.com/Login?Q=%2F")

{'label': 'benign',
 'class_id': 0,
 'score': 0.1750135058578919,
 'score_kind': 'uncalibrated_sigmoid',
 'calibrated': False,
 'threshold': 0.564855787905593,
 'model_version': 'student-week7-kd_mixed',
 'release_status': 'experimental',
 'android_verified': False}

Đọc tiếp: `docs/week7_student_protocol.md`, `docs/week7_student_handoff.md`,
`reports/week7_student.md`. Flow đóng gói đầy đủ nằm trong `scripts/train_student.py: main()`.